In [1]:
# Import required libraries
import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image
import ipywidgets as widgets
from IPython.display import clear_output

# Function to display images side by side
def show_images(images, titles):
    plt.figure(figsize=(15, 5))
    for i in range(len(images)):
        plt.subplot(1, len(images), i + 1)
        if len(images[i].shape) == 2:  # grayscale
            plt.imshow(images[i], cmap='gray')
        else:
            plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

# Image upload widget
uploader = widgets.FileUpload(accept='image/*', multiple=False)

# Dropdown for thresholding methods
thresholding_dropdown = widgets.Dropdown(
    options=['None', 'Otsu', 'Adaptive'],
    value='None',
    description='Threshold:',
)

# Button to trigger processing
process_button = widgets.Button(description='Process Image')

# Output widget
output = widgets.Output()

# Main processing function
def on_process_clicked(b):
    output.clear_output()
    if len(uploader.value) == 0:
        with output:
            print("Please upload an image.")
        return

    uploaded_file = next(iter(uploader.value.values()))
    content = uploaded_file['content']
    img_array = np.frombuffer(content, dtype=np.uint8)
    image = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    threshold_method = thresholding_dropdown.value

    if threshold_method == 'Otsu':
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif threshold_method == 'Adaptive':
        thresh = cv2.adaptiveThreshold(
            blurred, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            11, 2
        )
    else:
        thresh = blurred

    edges = cv2.Canny(thresh, 100, 200)

    with output:
        show_images([image, thresh, edges], ['Original', 'Thresholded', 'Canny Edges'])

# Bind button click
process_button.on_click(on_process_clicked)

# Display widgets
display(widgets.VBox([uploader, thresholding_dropdown, process_button, output]))
